# 第 4 讲实验：Attention Alternatives 与 MoE

本 Notebook 配合第 4 讲讲义。目标是验证 GQA 的 KV cache 账本与 causal mask，以及 MoE 的 top-k 路由、专家负载和辅助损失梯度。所有图均为本 Notebook 根据运行结果绘制的教学图。

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

START = Path.cwd().resolve()
REPO_ROOT = next((p for p in (START, *START.parents) if (p / "code" / "lecture_04").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("请在课程仓库内启动 Jupyter Lab")
sys.path.insert(0, str(REPO_ROOT / "code" / "lecture_04"))
from attention_moe_lab import GroupedQueryAttention, TopKMoE, kv_cache_bytes, expert_loads, load_balance_loss, count_parameters

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)
if device.type != "cuda":
    print("CPU fallback：只验证数值/形状，不报告 GPU 性能。")

## 实验 1：GQA 形状与 KV cache

GQA 主要减少 KV heads。注意力分数张量仍按 query heads 组织，因此 KV cache 缩小不意味着所有 attention FLOPs 都同比下降。

In [ ]:
torch.manual_seed(17)
x = torch.randn(2, 8, 32, device=device)
gqa = GroupedQueryAttention(d_model=32, n_query_heads=4, n_kv_heads=2).to(device).eval()
with torch.no_grad():
    gqa_out, weights = gqa(x)
print("input:", tuple(x.shape))
print("GQA output:", tuple(gqa_out.shape))
print("attention weights:", tuple(weights.shape))
common = dict(n_layers=12, batch_size=2, seq_len=128, head_dim=64, bytes_per_element=2)
gqa_bytes = kv_cache_bytes(**common, n_kv_heads=4)
mha_bytes = kv_cache_bytes(**common, n_kv_heads=16)
print(f"GQA KV cache: {gqa_bytes / 1024**2:.1f} MiB")
print(f"MHA KV cache: {mha_bytes / 1024**2:.1f} MiB")
assert gqa_out.shape == x.shape and weights.shape == (2, 4, 8, 8)
assert gqa_bytes == 3 * 1024**2 and mha_bytes == 4 * gqa_bytes

## 实验 2：因果注意力热图

上三角区域（query 位置早于 key 位置）应为零。这张图是根据当前随机输入和初始化权重生成的，不是论文中的性能图。

In [ ]:
heatmap = weights[0, 0].detach().cpu()
plt.figure(figsize=(5, 4))
plt.imshow(heatmap, vmin=0.0, cmap="viridis")
plt.colorbar(label="attention weight")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.title("GQA causal attention (sample 0, query head 0)")
plt.tight_layout()
plt.show()
assert torch.allclose(torch.triu(heatmap, diagonal=1), torch.zeros_like(heatmap), atol=1e-7)

## 实验 3：Top-k MoE 路由与专家负载

一个 token 选中 k 个专家，会产生 k 个 assignments。因此总 assignment 数应等于 token 数乘以 k。

In [ ]:
torch.manual_seed(23)
moe = TopKMoE(d_model=32, hidden_dim=64, n_experts=8, top_k=2).to(device)
tokens = torch.randn(2, 12, 32, device=device)
moe_out, routing = moe(tokens, return_routing=True)
loads = expert_loads(routing["topk_indices"], n_experts=8).detach().cpu()
print("MoE output:", tuple(moe_out.shape))
print("trainable parameters:", f"{count_parameters(moe):,}")
print("expert assignment counts:", loads.tolist())
print("assignment count sum:", int(loads.sum()), "expected:", tokens.shape[0] * tokens.shape[1] * 2)
assert moe_out.shape == tokens.shape
assert int(loads.sum()) == tokens.shape[0] * tokens.shape[1] * 2
assert torch.allclose(routing["topk_weights"].sum(-1), torch.ones(2, 12, device=device), atol=1e-6)
plt.figure(figsize=(7, 3))
plt.bar(range(8), loads.numpy())
plt.xlabel("Expert ID")
plt.ylabel("Top-k assignments")
plt.title("Observed expert load for a random batch")
plt.tight_layout()
plt.show()

## 实验 4：负载均衡 proxy 与梯度

这里使用 Switch-style 教学 proxy。不同论文和实现的辅助损失、归一化和系数可能不同，复现实验前应核对原论文。

In [ ]:
aux_loss = load_balance_loss(routing["router_probs"], routing["topk_indices"])
moe.zero_grad(set_to_none=True)
(moe_out.square().mean() + 0.01 * aux_loss).backward()
router_grad = moe.router.weight.grad
print(f"load-balance proxy: {aux_loss.item():.4f}")
print("router gradient finite:", router_grad is not None and torch.isfinite(router_grad).all().item())
print("expert gradients present:", any(p.grad is not None for e in moe.experts for p in e.parameters()))
assert torch.isfinite(aux_loss).item()
assert router_grad is not None and torch.isfinite(router_grad).all().item()

## 复盘

1. 为什么 KV cache 按 KV head 数缩放，但 attention weights 仍有 query-head 维度？
2. 为什么 assignment count 之和等于 token 数乘 top-k？
3. 当负载集中在少数专家时，哪一个专家可能限制 step latency？
4. 哪些实现细节仍缺少真实 GPU 证据？

请在实验记录中写下设备、PyTorch 版本、配置、观测结果与未验证项。